# Project Title
## E-Commerce Customer Behavior & Sales Analysis

# Objective
This project analyzes customer behavior and sales patterns in an e-commerce dataset.

Main goals:
- Understand customer purchasing behavior
- Clean and preprocess data for analysis
- Perform exploratory data analysis (EDA) with meaningful visualizations
- Segment customers using K-Means clustering
- Predict Customer Satisfaction using a Random Forest Regressor

# Import Libraries

In [ ]:
# Core libraries
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Visualization and notebook display
from IPython.display import display

# Machine learning and preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score,
)
import joblib

# Plot styles
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 6)

The required libraries are imported and plotting style is configured for consistent visual output.

# Load Dataset

In [ ]:
# Required placeholder path
file_path = "/content/ecommerce_customer_data.csv"

# Upload dataset in Colab if the file is not already present
if not os.path.exists(file_path):
    from google.colab import files
    import shutil

    print("Please upload your CSV dataset file.")
    uploaded = files.upload()

    if len(uploaded) == 0:
        raise FileNotFoundError("No file uploaded. Please upload a CSV file to continue.")

    uploaded_name = next(iter(uploaded))
    if not uploaded_name.lower().endswith(".csv"):
        raise ValueError("Uploaded file is not a CSV. Please upload a .csv file.")

    if uploaded_name != os.path.basename(file_path):
        shutil.move(uploaded_name, file_path)
        print(f"Uploaded '{uploaded_name}' and saved as '{file_path}'.")
    else:
        print(f"Uploaded file saved at: {file_path}")
else:
    print(f"Dataset already found at: {file_path}")

df = pd.read_csv(file_path)
print(f"Dataset loaded successfully from: {file_path}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())

The notebook now prompts a direct CSV upload in Colab and saves it to /content/ecommerce_customer_data.csv before loading it.

# Data Understanding

In [ ]:
# Basic structure and columns
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(list(df.columns))

# Data types and non-null counts
print("\nData types and non-null counts:")
df.info()

# Summary statistics
print("\nSummary statistics (including categorical columns):")
display(df.describe(include="all").transpose())

# Missing value overview
missing_counts = df.isna().sum().sort_values(ascending=False)
print("\nTop columns with missing values:")
display(missing_counts[missing_counts > 0].head(10))

This step gives an initial profile of the dataset, including data types, summary statistics, and missing values.

# Data Cleaning

In [ ]:
df_clean = df.copy()

# Clean Purchase_Amount by removing currency symbols and spaces, then convert to numeric
if "Purchase_Amount" in df_clean.columns:
    df_clean["Purchase_Amount"] = (
        df_clean["Purchase_Amount"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df_clean["Purchase_Amount"] = pd.to_numeric(df_clean["Purchase_Amount"], errors="coerce")

# Remove duplicate rows
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
removed_duplicates = initial_rows - len(df_clean)

# Handle missing values
# Numeric columns -> median
# Categorical columns -> mode
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
categorical_cols = df_clean.select_dtypes(exclude=[np.number]).columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    mode_value = df_clean[col].mode(dropna=True)
    if not mode_value.empty:
        df_clean[col] = df_clean[col].fillna(mode_value.iloc[0])
    else:
        df_clean[col] = df_clean[col].fillna("Unknown")

# Create a numeric income feature for analysis and clustering
income_col_candidates = [c for c in df_clean.columns if "income" in c.lower()]
if income_col_candidates:
    income_col = income_col_candidates[0]
    income_map = {"Low": 1, "Middle": 2, "Medium": 2, "High": 3, "Very High": 4}
    df_clean["Income_Numeric"] = df_clean[income_col].map(income_map)

    # Fallback encoding if mapping does not match categories
    if df_clean["Income_Numeric"].isna().all():
        temp_le = LabelEncoder()
        df_clean["Income_Numeric"] = temp_le.fit_transform(df_clean[income_col].astype(str))
    else:
        df_clean["Income_Numeric"] = df_clean["Income_Numeric"].fillna(df_clean["Income_Numeric"].median())

print(f"Removed duplicate rows: {removed_duplicates}")
print("Total missing values after cleaning:", int(df_clean.isna().sum().sum()))
display(df_clean.head())

Data cleaning completed: missing values handled, duplicate rows removed, and key numeric columns prepared for analysis.

# Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

# 1) Histogram of Purchase Amount
if "Purchase_Amount" in df_clean.columns:
    sns.histplot(df_clean["Purchase_Amount"], bins=30, kde=True, ax=axes[0], color="#2a9d8f")
    axes[0].set_title("Histogram of Purchase Amount")
else:
    axes[0].text(0.5, 0.5, "Purchase_Amount not found", ha="center", va="center")
    axes[0].set_title("Histogram of Purchase Amount")

# 2) Income vs Purchase Amount scatter plot
if "Income_Numeric" in df_clean.columns and "Purchase_Amount" in df_clean.columns:
    sns.scatterplot(data=df_clean, x="Income_Numeric", y="Purchase_Amount", alpha=0.6, ax=axes[1], color="#e76f51")
    axes[1].set_title("Income vs Purchase Amount")
    axes[1].set_xlabel("Income (Numeric)")
    axes[1].set_ylabel("Purchase Amount")
else:
    axes[1].text(0.5, 0.5, "Required columns not found", ha="center", va="center")
    axes[1].set_title("Income vs Purchase Amount")

# 3) Correlation heatmap
numeric_df = df_clean.select_dtypes(include=[np.number])
if numeric_df.shape[1] > 1:
    corr = numeric_df.corr()
    sns.heatmap(corr, cmap="YlGnBu", linewidths=0.5, ax=axes[2])
    axes[2].set_title("Correlation Heatmap")
else:
    axes[2].text(0.5, 0.5, "Not enough numeric columns", ha="center", va="center")
    axes[2].set_title("Correlation Heatmap")

# 4) Category-wise analysis
category_col = "Purchase_Category" if "Purchase_Category" in df_clean.columns else None
if category_col and "Purchase_Amount" in df_clean.columns:
    top_cat = (
        df_clean.groupby(category_col)["Purchase_Amount"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )
    sns.barplot(data=top_cat, x="Purchase_Amount", y=category_col, ax=axes[3], palette="viridis")
    axes[3].set_title("Top Categories by Average Purchase Amount")
else:
    axes[3].text(0.5, 0.5, "Category columns not found", ha="center", va="center")
    axes[3].set_title("Category-wise Analysis")

# 5) Additional insightful plot: Purchase Amount by Channel
channel_col = "Purchase_Channel" if "Purchase_Channel" in df_clean.columns else None
if channel_col and "Purchase_Amount" in df_clean.columns:
    sns.boxplot(data=df_clean, x=channel_col, y="Purchase_Amount", ax=axes[4], palette="Set3")
    axes[4].set_title("Purchase Amount by Channel")
    axes[4].tick_params(axis="x", rotation=25)
else:
    axes[4].text(0.5, 0.5, "Channel column not found", ha="center", va="center")
    axes[4].set_title("Purchase Amount by Channel")

# Extra insight plot (helps interpret the ML target)
if "Customer_Satisfaction" in df_clean.columns:
    sns.countplot(data=df_clean, x="Customer_Satisfaction", ax=axes[5], color="#264653")
    axes[5].set_title("Customer Satisfaction Distribution")
else:
    axes[5].text(0.5, 0.5, "Customer_Satisfaction not found", ha="center", va="center")
    axes[5].set_title("Customer Satisfaction Distribution")

plt.tight_layout()
plt.show()

EDA visualizations highlight spending patterns, income behavior, category performance, and satisfaction trends.

# Customer Segmentation (K-Means)

In [ ]:
# Select relevant numeric features for clustering
cluster_feature_candidates = [
    "Income_Numeric",
    "Purchase_Amount",
    "Frequency_of_Purchase",
    "Product_Rating",
    "Age"
]
cluster_features = [c for c in cluster_feature_candidates if c in df_clean.columns]

if len(cluster_features) < 2:
    raise ValueError("Not enough numeric features available for clustering.")

cluster_df = df_clean[cluster_features].copy().dropna()

# Standardize features before K-Means
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_df)

# Train K-Means model
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(cluster_scaled)

# Attach cluster labels back to the cleaned data subset
df_clustered = df_clean.loc[cluster_df.index].copy()
df_clustered["Cluster"] = cluster_labels

print("Cluster counts:")
print(df_clustered["Cluster"].value_counts().sort_index())

# Plot clusters using the first two selected features
x_feature = cluster_features[0]
y_feature = cluster_features[1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clustered, x=x_feature, y=y_feature, hue="Cluster", palette="tab10", s=60, alpha=0.8)
plt.title("Customer Segmentation using K-Means")
plt.legend(title="Cluster")
plt.show()

K-Means clustering groups similar customers, helping identify distinct behavior segments for targeted strategies.

# Machine Learning Model (Prediction)

In [ ]:
# Classification-ready dataset for report outputs
df_model = df.copy()

# Save raw dataset overview metrics for Table 1
dataset_rows = df_model.shape[0]
dataset_cols = df_model.shape[1]
dataset_duplicates = int(df_model.duplicated().sum())
dataset_missing = int(df_model.isna().sum().sum())

# Clean key numeric fields
if "Purchase_Amount" in df_model.columns:
    df_model["Purchase_Amount"] = (
        df_model["Purchase_Amount"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df_model["Purchase_Amount"] = pd.to_numeric(df_model["Purchase_Amount"], errors="coerce")

if "Customer_Satisfaction" not in df_model.columns:
    raise ValueError("Customer_Satisfaction column not found in dataset.")

df_model["Customer_Satisfaction"] = pd.to_numeric(df_model["Customer_Satisfaction"], errors="coerce")

# Remove duplicates
df_model = df_model.drop_duplicates().reset_index(drop=True)

# Normalize binary-style features used in Figure 2
binary_feature_candidates = ["Discount_Used", "Customer_Loyalty_Program_Member", "Return_Rate"]
binary_map = {
    "true": 1,
    "false": 0,
    "yes": 1,
    "no": 0,
    "1": 1,
    "0": 0,
}

for col in binary_feature_candidates:
    if col in df_model.columns:
        as_text = df_model[col].astype(str).str.strip().str.lower()
        mapped = as_text.map(binary_map)
        if mapped.notna().sum() > 0:
            fill_bin = int(mapped.mode().iloc[0]) if not mapped.mode().empty else 0
            df_model[col] = mapped.fillna(fill_bin).astype(int)
        else:
            as_num = pd.to_numeric(df_model[col], errors="coerce")
            if as_num.notna().sum() > 0:
                df_model[col] = (as_num > 0).astype(int)

# Handle missing values
numeric_cols = df_model.select_dtypes(include=[np.number]).columns
categorical_cols = df_model.select_dtypes(exclude=[np.number]).columns

for col in numeric_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

for col in categorical_cols:
    mode_value = df_model[col].mode(dropna=True)
    df_model[col] = df_model[col].fillna(mode_value.iloc[0] if not mode_value.empty else "unknown")

# Build binary class target from Customer_Satisfaction
satisfaction_threshold = df_model["Customer_Satisfaction"].median()
df_model["satisfaction_class"] = (df_model["Customer_Satisfaction"] >= satisfaction_threshold).astype(int)

if df_model["satisfaction_class"].nunique() < 2:
    raise ValueError("Target class has fewer than 2 classes. Cannot train classification models.")

# Table 2 source (feature description)
table2_feature_description = pd.DataFrame(
    {
        "Feature": df_model.columns,
        "Data Type": df_model.dtypes.astype(str).values,
        "Missing Values": df_model.isna().sum().values,
        "Unique Values": df_model.nunique().values,
    }
)

# Prepare features and target
X = df_model.drop(columns=["Customer_Satisfaction", "satisfaction_class", "Customer_ID"], errors="ignore").copy()
y = df_model["satisfaction_class"].copy()

# Encode categorical predictor columns
label_encoders = {}
for col in X.select_dtypes(exclude=[np.number]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Final numeric safety
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].fillna(X[col].median())

# Train-test split (80/20)
stratify_target = y if y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_target,
)

# Train models for comparison
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
)

lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

# Predictions and probabilities
y_pred_lr = lr_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Classification models trained successfully.")
print("Training samples:", X_train.shape[0], "| Testing samples:", X_test.shape[0])
print("Feature count:", X.shape[1])

Logistic Regression and Random Forest Classifier are trained on customer satisfaction classes for comparison and report-ready evaluation.

# Results & Evaluation

In [ ]:
# Table 1: Dataset Overview
table1_dataset_overview = pd.DataFrame(
    {
        "Metric": ["Rows", "Columns", "Duplicate Rows", "Missing Values"],
        "Value": [dataset_rows, dataset_cols, dataset_duplicates, dataset_missing],
    }
)
print("Table 1: Dataset Overview")
display(table1_dataset_overview)

# Table 2: Feature Description
print("\nTable 2: Feature Description")
display(table2_feature_description)

# Safe AUC helper (prevents crash when test set has one class)
def safe_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

auc_lr_val = safe_auc(y_test, y_prob_lr)
auc_rf_val = safe_auc(y_test, y_prob_rf)

# Table 3: Model Performance Metrics
table3_model_performance_metrics = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression",
            "Accuracy": accuracy_score(y_test, y_pred_lr),
            "Precision": precision_score(y_test, y_pred_lr, zero_division=0),
            "Recall": recall_score(y_test, y_pred_lr, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred_lr, zero_division=0),
            "AUC": auc_lr_val,
        },
        {
            "Model": "Random Forest",
            "Accuracy": accuracy_score(y_test, y_pred_rf),
            "Precision": precision_score(y_test, y_pred_rf, zero_division=0),
            "Recall": recall_score(y_test, y_pred_rf, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred_rf, zero_division=0),
            "AUC": auc_rf_val,
        },
    ]
)
print("\nTable 3: Model Performance Metrics")
display(table3_model_performance_metrics.round(4))

# Table 4: Final Model Comparison
table4_final_model_comparison = (
    table3_model_performance_metrics
    .sort_values(by=["F1 Score", "AUC"], ascending=False, na_position="last")
    .reset_index(drop=True)
)
table4_final_model_comparison.insert(0, "Rank", np.arange(1, len(table4_final_model_comparison) + 1))
print("\nTable 4: Final Model Comparison")
display(table4_final_model_comparison.round(4))

print("\nClassification Report (Random Forest):")
print(classification_report(y_test, y_pred_rf, zero_division=0))

# Figure 1: Class Distribution of Customer Satisfaction Classes
plt.figure(figsize=(7, 4))
sns.countplot(x="satisfaction_class", data=df_model, palette="Set2")
plt.title("Figure 1: Class Distribution of Customer Satisfaction Classes")
plt.xlabel("Class (0 = Low Satisfaction, 1 = High Satisfaction)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Figure 2: Binary Feature Analysis (Discount Used, Loyalty Program, Return Behavior)
binary_plot_cols = [c for c in binary_feature_candidates if c in df_model.columns]
if len(binary_plot_cols) == 0:
    binary_plot_cols = [c for c in X.columns if X[c].nunique() <= 2][:3]

plt.figure(figsize=(9, 4))
if len(binary_plot_cols) > 0:
    binary_plot_df = df_model[binary_plot_cols].copy().melt(var_name="Feature", value_name="Value")
    binary_plot_df["Value"] = binary_plot_df["Value"].astype(str)
    sns.countplot(data=binary_plot_df, x="Feature", hue="Value", palette="Paired")
    plt.xlabel("Binary Features")
    plt.ylabel("Count")
else:
    fallback_df = pd.DataFrame(
        {
            "Feature": ["satisfaction_class", "satisfaction_class"],
            "Value": ["0", "1"],
            "Count": [
                int((df_model["satisfaction_class"] == 0).sum()),
                int((df_model["satisfaction_class"] == 1).sum()),
            ],
        }
    )
    sns.barplot(data=fallback_df, x="Feature", y="Count", hue="Value", palette="Paired")
    plt.xlabel("Fallback Binary Feature")
    plt.ylabel("Count")
plt.title("Figure 2: Binary Feature Analysis (Discount Used, Loyalty Program, Return Behavior)")
plt.tight_layout()
plt.show()

# Figure 3: Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Class 0", "Class 1"],
    yticklabels=["Class 0", "Class 1"],
)
plt.title("Figure 3: Confusion Matrix (Customer Satisfaction Classifier)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Figure 4: Model Performance Comparison
plot_perf = table3_model_performance_metrics.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1 Score", "AUC"],
    var_name="Metric",
    value_name="Score",
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_perf, x="Metric", y="Score", hue="Model")
plt.title("Figure 4: Model Performance Comparison")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Figure 5: ROC Curve for Customer Satisfaction Classification
plt.figure(figsize=(7, 5))
if len(np.unique(y_test)) > 1:
    fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
    fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
    auc_lr = auc(fpr_lr, tpr_lr)
    auc_rf = auc(fpr_rf, tpr_rf)

    plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC={auc_lr:.3f})")
    plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={auc_rf:.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
else:
    plt.plot([0, 1], [0, 1], "k--")
    plt.text(0.15, 0.5, "ROC not available: only one class in test set")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
plt.title("Figure 5: ROC Curve (Customer Satisfaction Classifier)")
plt.tight_layout()
plt.show()

# Figure 6: Feature Importance for Customer Satisfaction Prediction
feature_importance_df = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": rf_model.feature_importances_,
    }
).sort_values(by="Importance", ascending=False).head(12)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_df, x="Importance", y="Feature", color="#4C72B0")
plt.title("Figure 6: Feature Importance for Customer Satisfaction Prediction")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Save and download the final model
model_path = "/content/final_random_forest_classifier.pkl"
joblib.dump(rf_model, model_path)
print(f"\nFinal model saved at: {model_path}")

try:
    from google.colab import files
    files.download(model_path)
    print("Model download started.")
except Exception:
    print("Run in Google Colab to download the model automatically.")

This section generates all required report outputs in flow: Table 1 to Table 4 and Figure 1 to Figure 6, plus a downloadable final model.

# Conclusion
This notebook now follows a direct report workflow in normal execution order.

Key outcomes:
- Data is cleaned and prepared for classification.
- Logistic Regression and Random Forest models are trained and compared.
- Required outputs are generated in-flow:
  - Table 1 to Table 4
  - Figure 1 to Figure 6
- The final Random Forest model is exported and can be downloaded in Colab.

This setup is ready for taking screenshots directly during notebook execution for report submission.